# Football Results Analyser

This notebook scrapes recent match results from **Flashscore.com** across multiple leagues using Selenium (required because the site is fully JavaScript-rendered — plain `requests` won't work), then feeds the data to an OpenAI model for a narrative analysis.

**Leagues covered:** EPL · La Liga · Bundesliga

**What it demonstrates:**
- Selenium for JS-heavy sites (addresses the limitation called out in `day1.ipynb`)
- Structuring scraped data as a pandas DataFrame before passing to an LLM
- Using a system prompt to steer the tone of analysis
- Displaying LLM output as rendered Markdown

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

## Step 1 — Scrape results

The scraper runs a headless Chrome browser, dismisses the cookie banner, and waits for the match elements to load before extracting them. Set `headless=False` if you want to watch it work.

In [ ]:
from scraper import scrape_all_leagues

rows = scrape_all_leagues(headless=True)
df = pd.DataFrame(rows, columns=['league', 'date', 'home_team', 'score', 'away_team'])
print(f'Total matches scraped: {len(df)}')
df.head(10)

## Step 2 — Results per league

In [ ]:
for league, group in df.groupby('league', sort=False):
    print(f'\n=== {league} ({len(group)} matches) ===')
    print(group[['date', 'home_team', 'score', 'away_team']].to_string(index=False))

## Step 3 — LLM Analysis

We format the full results table as plain text and ask GPT to identify standout results, upsets, and scoring trends across all leagues.

In [ ]:
system_prompt = """
You are an expert football analyst covering European football.
You will be given a batch of recent match results from multiple leagues.
Your job is to write a concise, insightful analysis that covers:
- The most surprising upsets or high-scoring games
- Any team that looks to be in outstanding or poor form
- A brief cross-league comparison of attacking play (goals per game)
Write in an engaging style. Respond in Markdown. Do not wrap the markdown in a code block.
"""

def build_results_text(df):
    lines = []
    for league, group in df.groupby('league', sort=False):
        lines.append(f'### {league}')
        for _, row in group.iterrows():
            lines.append(f"  {row['date']}  {row['home_team']} {row['score']} {row['away_team']}")
        lines.append('')
    return '\n'.join(lines)

def analyse_results(df):
    results_text = build_results_text(df)
    response = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f'Here are the latest results:\n\n{results_text}'}
        ]
    )
    return response.choices[0].message.content

In [ ]:
analysis = analyse_results(df)
display(Markdown(analysis))

## Step 4 — Per-league breakdown

Ask the model to focus on one league at a time for a deeper read.

In [ ]:
def analyse_league(df, league_name):
    league_df = df[df['league'] == league_name]
    if league_df.empty:
        print(f'No data for {league_name}')
        return
    results_text = build_results_text(league_df)
    response = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f'Focus only on {league_name}. Here are the results:\n\n{results_text}'}
        ]
    )
    display(Markdown(response.choices[0].message.content))

analyse_league(df, 'EPL')

In [ ]:
analyse_league(df, 'La Liga')

In [ ]:
analyse_league(df, 'Bundesliga')

## Extending this notebook

Some ideas to take this further:

- **Add more leagues** — extend the `LEAGUES` dict in `scraper.py` with Serie A, Ligue 1, etc.
- **Predict upcoming fixtures** — scrape the fixtures page instead of results and ask GPT to predict outcomes
- **Track form over time** — run the scraper on a schedule and store results in a CSV to feed the model historical context
- **Export the analysis** — pipe the Markdown output into the PDF generator from the original project